In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.fft import fft, fftfreq

# --- parâmetros físicos ---
eps = 8.854187817e-12  # Vacuum permittivity (F/m)
mu = 4 * np.pi * 1e-7  # Vacuum permeability (H/m)
c = 1 / np.sqrt(eps * mu)

# Dimensões da cavidade
a, b, c = 0.05, 0.04, 0.03

# Discretização
Nx, Ny, Nz = 200, 160, 120

dx = a / Nx
dy = b / Ny
dz = c / Nz

# CFL 3D
dt = 0.99 / (c * np.sqrt((1/dx**2)+(1/dy**2)+(1/dz**2)))

Nt = 500

# ------------------------
# Campos (4D: t, x, y, z)
# ------------------------
Ex = np.zeros((Nx, Ny, Nz))
Ey = np.zeros((Nx, Ny, Nz))
Ez = np.zeros((Nx, Ny, Nz))

Hx = np.zeros((Nx, Ny, Nz))
Hy = np.zeros((Nx, Ny, Nz))
Hz = np.zeros((Nx, Ny, Nz))

# ------------------------
# Fonte (pulso gaussiano)
# ------------------------
def source(t):
    return np.exp(-((t-30)/10)**2)

ix, iy, iz = Nx//2, Ny//2, Nz//2 # centro da cavidade, onde tem a fonte

signal = np.zeros(Nt)

# ------------------------
# Loop no tempo (Yee 3D)
# ------------------------
for n in range(Nt-1):

# --- Update H ---
    Hx[:, :-1, :-1] -= (dt/mu)*(
        (Ez[:, 1:, :-1] - Ez[:, :-1, :-1])/dy -
        (Ey[:, :-1, 1:] - Ey[:, :-1, :-1])/dz
    )

    Hy[:-1, :, :-1] -= (dt/mu)*(
        (Ex[:-1, :, 1:] - Ex[:-1, :, :-1])/dz -
        (Ez[1:, :, :-1] - Ez[:-1, :, :-1])/dx
    )

    Hz[:-1, :-1, :] -= (dt/mu)*(
        (Ey[1:, :-1, :] - Ey[:-1, :-1, :])/dx -
        (Ex[:-1, 1:, :] - Ex[:-1, :-1, :])/dy
    )

    # --- Update E ---
    Ex[1:-1, 1:-1, 1:-1] += (dt/eps)*(
        (Hz[1:-1, 1:-1, 1:-1] - Hz[1:-1, :-2, 1:-1])/dy -
        (Hy[1:-1, 1:-1, 1:-1] - Hy[1:-1, 1:-1, :-2])/dz
    )

    Ey[1:-1, 1:-1, 1:-1] += (dt/eps)*(
        (Hx[1:-1, 1:-1, 1:-1] - Hx[1:-1, 1:-1, :-2])/dz -
        (Hz[1:-1, 1:-1, 1:-1] - Hz[:-2, 1:-1, 1:-1])/dx
    )

    Ez[1:-1, 1:-1, 1:-1] += (dt/eps)*(
        (Hy[1:-1, 1:-1, 1:-1] - Hy[:-2, 1:-1, 1:-1])/dx -
        (Hx[1:-1, 1:-1, 1:-1] - Hx[1:-1, :-2, 1:-1])/dy
    )

    # --- Source ---
    Ez[ix, iy, iz] += source(n)

    # --- PEC boundary ---
    Ex[0,:,:] = Ex[-1,:,:] = 0
    Ey[:,0,:] = Ey[:,-1,:] = 0
    Ez[:,:,0] = Ez[:,:,-1] = 0

    # --- Record probe ---
    signal[n] = Ez[ix, iy, iz]

fft = np.fft.fft(signal)
freq = np.fft.fftfreq(Nt, dt)

# Só frequências positivas
mask = freq > 0

plt.plot(freq[mask], np.abs(fft[mask]))
plt.xlabel("Frequência (Hz)")
plt.ylabel("Amplitude")
plt.title("Espectro de frequências da cavidade")
plt.show()

/tmp/ipykernel_5419/3081830460.py:53: RuntimeWarning: overflow encountered in multiply
  Hx[:, :-1, :-1] -= (dt/mu)*(
/tmp/ipykernel_5419/3081830460.py:58: RuntimeWarning: overflow encountered in multiply
  Hy[:-1, :, :-1] -= (dt/mu)*(
/tmp/ipykernel_5419/3081830460.py:71: RuntimeWarning: overflow encountered in subtract
  (Hy[1:-1, 1:-1, 1:-1] - Hy[1:-1, 1:-1, :-2])/dz
/tmp/ipykernel_5419/3081830460.py:71: RuntimeWarning: overflow encountered in divide
  (Hy[1:-1, 1:-1, 1:-1] - Hy[1:-1, 1:-1, :-2])/dz
/tmp/ipykernel_5419/3081830460.py:69: RuntimeWarning: overflow encountered in multiply
  Ex[1:-1, 1:-1, 1:-1] += (dt/eps)*(
/tmp/ipykernel_5419/3081830460.py:75: RuntimeWarning: overflow encountered in subtract
  (Hx[1:-1, 1:-1, 1:-1] - Hx[1:-1, 1:-1, :-2])/dz -
/tmp/ipykernel_5419/3081830460.py:75: RuntimeWarning: overflow encountered in divide
  (Hx[1:-1, 1:-1, 1:-1] - Hx[1:-1, 1:-1, :-2])/dz -
/tmp/ipykernel_5419/3081830460.py:74: RuntimeWarning: overflow encountered in multiply
  Ey[

KeyboardInterrupt: 